In [3]:
import plotly.express as px
import os
# os.chdir(f'/home/cloudcraftz/Cloudcraftz/HFT/alpha_v07/F_Intraday')
import pandas as pd
import numpy as np
# import humanize
import math
import glob
import os
import re
import subprocess
from datetime import datetime
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from datetime import datetime, time, timedelta
# from option_trading_plots import Chartpack
# from modules._logger import get_exception_line_no

# Enhnancement Sheet Maker

In [2]:
selected_columns = [
        'Strategy',
        'Underlying',
        'Short Description',
        'Lot Size',
        'Year',
        'Code Base',
        'Total P/L',
        'Avg. Monthly PnL',
        'Max Monthly Gain',
        'Max Monthly Loss',
        'SD Monthly PnL',
        'Monthly Avg. PnL / SD',
        'Peak Drawdown',
        'Ratio (Total P/L / Net Premium)',
        'Execution_Date',
    ]

def excel_to_dict(file_path):
    result_dict = {}
    # Read Excel file into a pandas DataFrame
    df = pd.read_excel(file_path)
    # Assuming the first row is a header, we skip it
    for index, row in df.iterrows():
        for i in range(0, len(row), 2):
            if i + 1 < len(row):
                result_dict[row[i]] = row[i + 1]
    return result_dict
2
def convert_to_number(value):
    try:
        # Remove non-numeric characters and convert to float
        return float(value.replace(',', '').split()[0])
    except ValueError:
        return value  # Return the original value if conversion fails

def extract_year(text):
    pattern = r'\b(19\d{2}|20\d{2})\b'
    match = re.search(pattern, text)
    if match:
        return match.group(0)
    return match

def process_technical_summary(prefix,code_base,ul,y,s,lot_size = 100):


    main_folder_path = prefix
    print(main_folder_path)

    list_dicts = []

    # Iterate through each folder in the main folder
    folder_names = os.listdir(main_folder_path)
    folder_names.sort()
    for folder_name in folder_names:
        print(folder_name)
        folder_path = os.path.join(main_folder_path, folder_name, 'consolidated_store')
        if os.path.isdir(folder_path):
            # print("Reading files from folder:", folder_name)
            # Iterate through files in the current folder
            for file_name in os.listdir(folder_path):
                if file_name.endswith('.xlsx'):
                    file_path = os.path.join(folder_path, file_name)
                    # Read the YAML file
                    excel_dict = excel_to_dict(file_path)
                    excel_dict['Strategy'] = s
                    excel_dict['Year'] = str(int(folder_name.split('_')[-1]))
                    excel_dict['Underlying'] = ul
                    excel_dict['Lot Size'] = lot_size
                    excel_dict['Code Base'] = code_base
                    excel_dict['Short Description'] = folder_name
                    excel_dict['Execution_Date'] = datetime.now().date()
                    converted_data = {key: convert_to_number(str(value)) for key, value in excel_dict.items()}
                    
                    list_dicts.append(converted_data)

    df = pd.DataFrame(list_dicts)
    df_selected = df[selected_columns]
    return df_selected

In [15]:
def calculate_adjusted_pnl(combined_intraday_df):
    try:
        # print(combined_intraday_df.head())
        # Group the time durations wherein the straddle is (in)active
        combined_intraday_df['is_active'] = combined_intraday_df['Portfolio Delta']!=0
        combined_intraday_df['Group'] = (combined_intraday_df['is_active'] != combined_intraday_df['is_active'].shift()).cumsum()
        combined_intraday_df['Timestamp'] = combined_intraday_df.index
        combined_intraday_df['Spot_Change'] = combined_intraday_df['Spot'].pct_change()
        
        # Find the P/L per trade by time of entry
        columns = ['Values','Timestamp','is_active', 'Group','Spot','Spot_Change']
        resampled_df = combined_intraday_df[columns].groupby('Group').first()
        resampled_df['straddlewise_pnl'] = resampled_df['Values'].diff().shift(-1)
        resampled_df['residual_pnl'] = resampled_df['straddlewise_pnl'].shift(1)
        resampled_df.loc[~resampled_df['is_active'],'residual_pnl'] = 0
        resampled_df['adjusted_pnl'] = resampled_df['straddlewise_pnl'] + resampled_df['residual_pnl']
        resampled_df.loc[~resampled_df['is_active'],'adjusted_pnl'] = 0
        resampled_df['adjusted_pnl'].iloc[0] = resampled_df['straddlewise_pnl'].iloc[0]

        return resampled_df,combined_intraday_df
    except Exception as e:
        print(f'Error in calculate_adjusted_pnl : {e} at line : {get_exception_line_no()}')

def get_trade_packet_information(main_folder_path,folder_name):
    try:
        
        year_filepath = os.path.join(main_folder_path,folder_name,'consolidated_store/backtest')
        chrtpck = Chartpack(filepath=year_filepath, name=Strategy, asset_class="Options", underlying=underlying,
                            startMonth="Jan", startYear=str(year), endMonth="Dec", endYear=str(year),
                            graph_width=1400, graph_height=450, graph_font_size=12)
        intraday_df = chrtpck.getData()
        intraday_df = intraday_df.set_index('Timestamp')
        backup_df,combined_intraday_df = calculate_adjusted_pnl(intraday_df)
        combined_intraday_df['Timestamp'] = pd.to_datetime(combined_intraday_df['Timestamp'])
        # display(combined_intraday_df.head())
        # display(backup_df.head())
        trade_time_df = pd.DataFrame()
        for idx in range(len(backup_df)):
            if backup_df.iloc[idx]['is_active'] == True:
                trade_time_df.at[idx,'Trade_Start'] = backup_df.iloc[idx]['Timestamp']
                if backup_df.iloc[idx+1]['is_active'] == False:
                    trade_time_df.at[idx,'Trade_End'] = backup_df.iloc[idx+1]['Timestamp']
                    trade_time_df.at[idx,'Trade_pnl'] = backup_df.iloc[idx]['adjusted_pnl']

        # print(trade_time_df.head())
        trade_time_df['Trade_packet_time'] = [(trade_time_df.iloc[x]['Trade_End'] - trade_time_df.iloc[x]['Trade_Start']).total_seconds()//60 for x in range(len(trade_time_df))]
        trade_time_df['Date'] = trade_time_df['Trade_Start'].dt.date

        short_trades = pd.DataFrame()
        yearly_pnl = trade_time_df['Trade_pnl'].sum()
        # Short trades are trade those that have a duration of <= 5
        for date in trade_time_df['Date'].unique():
            date_df = trade_time_df[trade_time_df['Date']== date]
            
            short_trades.at[date,'Pnl_of_short_trades'] = date_df[date_df['Trade_packet_time'] <= 5]['Trade_pnl'].sum()
            short_trades.at[date,'Pnl_pct_of_short_trades'] = (date_df[date_df['Trade_packet_time'] <= 5]['Trade_pnl'].sum() / yearly_pnl)*100
            short_trades.at[date,'Pct_shrt_trade_in_mkt'] = (date_df[date_df['Trade_packet_time'] <= 5]['Trade_packet_time'].sum() / date_df['Trade_packet_time'].sum())*100
            short_trades.at[date,'Shrt_trade_count'] = len(date_df[date_df['Trade_packet_time'] <= 5])
            short_trades.at[date,'Shrt_trade_total_duration'] = date_df[date_df['Trade_packet_time'] <= 5]['Trade_packet_time'].sum()

        return short_trades,trade_time_df
        
    except Exception as e:
        print(f'Error in get_trade_packet_information : {e} at line : {get_exception_line_no()}')
        print(folder_name)

def process_vssma_summary(prefix):
    try:
        main_folder_path = prefix
        data_count = {'2020' : 237 , '2021' : 226 , '2022' : 231 , '2023' : 242, '2024' : 110}
        folder_names = [f for f in os.listdir(main_folder_path) if os.path.isdir(os.path.join(main_folder_path, f))]
        folder_names = sorted(folder_names)
        vssma_signals = []
        trades_hit = []
        for folder_name in folder_names:
            short_trades,trade_time_df = get_trade_packet_information(main_folder_path,folder_name)
            count_shrt_trade = (short_trades['Shrt_trade_count'].mean())
            total_short_trade = (short_trades['Shrt_trade_count'].sum())
            pct_shrt_trade_in_mkt = (short_trades['Pct_shrt_trade_in_mkt'].mean())/100
            vssma_signals.append([folder_name,count_shrt_trade,total_short_trade,pct_shrt_trade_in_mkt])

            log_dir = f"{main_folder_path}/{folder_name}/log/"
            log_files = glob.glob(os.path.join(log_dir, "*.log"))
            
            if log_files:  # Only proceed if log files are found
                input_files = " ".join(log_files)  # Join the file paths with a space
                output_path = f"{log_dir}max_trades_hit.csv"
                
                command = f'cat {input_files} | grep "No trade quota left. Trade Count :" > "{output_path}"'
                # Execute the command
                subprocess.run(command, shell=True)
                
                # Read and process the output CSV file
                df = pd.read_csv(output_path, names=['C1', 'C2', 'C3'])
                df['Timestamp'] = [x.split('::')[-1][1:] for x in df['C2']]
                df['Max Trade Count'] = [x.split('Trade Count : ')[1] for x in df['C3']]
                
                df = df[['Timestamp', 'Max Trade Count']]
                df.set_index('Timestamp', inplace=True)
                df.index = pd.to_datetime(df.index)
                days_hit = df.resample('D').first().dropna().shape[0]
                days_hit_pct = (days_hit/data_count[year])
                
                trades_hit.append([folder_name, days_hit_pct])

        results_df = pd.DataFrame(vssma_signals, columns=['Short Description','Mean Short Trades', 'Total Short Trades', 'Short trades in Market Time']).set_index('Short Description')
        trades_hit_df = pd.DataFrame(trades_hit, columns=['Short Description', 'Total Max Trade hit']).set_index('Short Description')
        results_df = pd.merge(results_df, trades_hit_df, on='Short Description')
        return results_df
    except Exception as e:
        print(f'Error in process_vssma_summary : {e} at line : {get_exception_line_no()}')


In [16]:
# year = '2023'
# prefix = f'/home/cloudcraftz/Cloudcraftz/HFT/alpha_v07/F_Intraday/sig_sample/outputs/NIFTY_VSSMA_CUSTOMER/Quick_Action/{year}/max_trades_10'
# folder_names = [f for f in os.listdir(prefix) if os.path.isdir(os.path.join(prefix, f))]

# folder_names = sorted(folder_names)
# trades_hit = []
# for folder_name in folder_names:
#     log_dir = f"{prefix}/{folder_name}/log/"
#     log_files = glob.glob(os.path.join(log_dir, "*.log"))
    
#     if log_files:  # Only proceed if log files are found
#         input_files = " ".join(log_files)  # Join the file paths with a space
#         output_path = f"{log_dir}max_trades_hit.csv"
        
#         command = f'cat {input_files} | grep "No trade quota left. Trade Count :" > "{output_path}"'
#         # Execute the command
#         subprocess.run(command, shell=True)
        
#         # Read and process the output CSV file
#         df = pd.read_csv(output_path, names=['C1', 'C2', 'C3'])
#         df['Timestamp'] = [x.split('::')[-1][1:] for x in df['C2']]
#         df['Max Trade Count'] = [x.split('Trade Count : ')[1] for x in df['C3']]
        
#         df = df[['Timestamp', 'Max Trade Count']]
#         df.set_index('Timestamp', inplace=True)
#         df.index = pd.to_datetime(df.index)
#         days_hit = df.resample('D').first().dropna().shape[0]
#         days_hit_pct = (days_hit/data_count[year])
        
#         trades_hit.append([folder_name, days_hit_pct])
    
# trades_hit_df = pd.DataFrame(trades_hit, columns=['Short Description', 'Total Max Trade hit']).set_index('Short Description')

In [17]:
def process_trade_summary_files(file_path):
    file_names = np.array(glob.glob(pathname=os.path.join(os.path.join(file_path,'consolidated_store','blotter','summary'),"*.csv")))
    file_names.sort(kind="stable")
    
    trade_summary_df = pd.concat([pd.read_csv(file_path) for file_path in file_names], axis=0) // 2
    trade_summary_df.drop('nunwinds', axis=1, inplace=True)
    if 'Unnamed: 0' in list(trade_summary_df.columns):
        trade_summary_df.drop('Unnamed: 0', axis=1, inplace=True)
    
    dates = [os.path.basename(file_name).split("_")[2][:8] for file_name in file_names]
    trade_summary_df['Date'] = dates
    trade_summary_df['Date'] = pd.to_datetime(trade_summary_df['Date'])
    trade_summary_df.set_index('Date', inplace=True)
    
    return trade_summary_df

def calculate_time_in_market(file_path):
    time_in_market = []
    file_names = np.array(glob.glob(pathname=os.path.join(os.path.join(file_path,'consolidated_store', 'backtest'),"*.csv")))
    file_names.sort(kind="stable")
    for file in file_names:
        df = pd.read_csv(file)
        t = df[df['Portfolio Delta']!=0].__len__()
        time_in_market.append(t)
    return time_in_market

def generate_trade_summary_df(file_path):
    trade_summary_df = process_trade_summary_files(file_path)
    trade_summary_df['Time in Market'] = calculate_time_in_market(file_path)
    return trade_summary_df

def calculate_pnl(file_path):
    csv_file_path = os.path.join(file_path,'consolidated_store', 'technical_summary.csv')
    df = pd.read_csv(csv_file_path)
    value_with_suffix = df.iloc[0, 1]
    value_without_suffix = value_with_suffix.replace('(INR)', '').replace(',', '')
    value_as_int = int(value_without_suffix)
    return value_as_int

rename_dict = {
        'ntrades' : 'Daily Strategy Trades',
        'nunwinds' : 'Daily Unwind Trades',
        'nhedges' : 'Daily Hedge Trades',
        'Time in Market' : 'Time in Market (mins)'
    }

def process_blotter_summary(path,weekly_stats):
    results = []  
    folder_names = os.listdir(path)
    folder_names.sort()
    for folder_name in folder_names:
        folder_path = os.path.join(path, folder_name)
        # print("process_blotter_summary",folder_name)
        if os.path.isdir(folder_path):
            trade_summary_df = generate_trade_summary_df(folder_path)
            trade_summary_df.rename(inplace=True, columns=rename_dict)
            
            # Calculate the three values and append them to the results list
            if weekly_stats:
                trade_summary_df.index = pd.to_datetime(trade_summary_df.index)
                trade_summary_df = trade_summary_df.resample('W').sum()
                median_trades = trade_summary_df['Daily Strategy Trades'].median()
                max_trades = trade_summary_df['Daily Strategy Trades'].max()
                hedge_trades = trade_summary_df['Daily Hedge Trades'].median()
                time_in_market = trade_summary_df['Time in Market (mins)'].mean()
                results.append([folder_name, median_trades, max_trades, hedge_trades,time_in_market])
            else:
                total_trades = trade_summary_df['Daily Strategy Trades'].sum()
                median_trades = trade_summary_df['Daily Strategy Trades'].median()
                max_trades = trade_summary_df['Daily Strategy Trades'].max()
                hedge_trades = trade_summary_df['Daily Hedge Trades'].median()
                time_in_market = trade_summary_df['Time in Market (mins)'].mean()
                results.append([folder_name, median_trades, max_trades, hedge_trades,total_trades,time_in_market])
    
    
    results_df = pd.DataFrame(results, columns=['Short Description','Median Trades', 'Max Trades', 'Hedge Trades','Total Strategy Trades','Time in Market']).set_index('Short Description')
    return results_df


In [23]:
prefix = '/home/cloudcraftz/Cloudcraftz/HFT/alpha_v07/F_Intraday/sig_sample/outputs/NIFTY_221_CUSTOMER/STOP_METHOD/2024/400000'
code_base = 'future_intraday'
underlying = 'NIFTY'
Strategy = 'Straddle'
year = '2024'
weekly_stats = False
GET_BLOTTER_SUMMARY = False
GET_VSSMA_SUMMARY = False

output_summary = process_technical_summary(prefix,code_base, underlying,year,Strategy,lot_size=25)
if GET_BLOTTER_SUMMARY:
    blotter_summary = process_blotter_summary(prefix,weekly_stats)
    output_summary = pd.merge(output_summary, blotter_summary, on='Short Description')
else:
    pass

if GET_VSSMA_SUMMARY:
    vssma_summary = process_vssma_summary(prefix)
    output_summary = pd.merge(output_summary, vssma_summary, on='Short Description')
else:
    pass
output_summary.to_csv(os.path.join(prefix,'output.csv'), index=False)
display(output_summary)


/home/cloudcraftz/Cloudcraftz/HFT/alpha_v07/F_Intraday/sig_sample/outputs/NIFTY_221_CUSTOMER/STOP_METHOD/2024/400000
NIFTY_221_400000_-400000_2024
NIFTY_221_400000_-500000_2024
NIFTY_221_400000_-600000_2024
NIFTY_221_400000_-700000_2024


,Strategy,Underlying,Short Description,Lot Size,Year,Code Base,Total P/L,Avg. Monthly PnL,Max Monthly Gain,Max Monthly Loss,SD Monthly PnL,Monthly Avg. PnL / SD,Peak Drawdown,Ratio (Total P/L / Net Premium),Execution_Date
0,Straddle,NIFTY,NIFTY_221_400000_-400000_2024,25.0,2024.0,future_intraday,14495015.0,1811877.0,3585231.0,-499680.0,1333377.0,1.36,-3991620.0,9.05%,2024-09-06
1,Straddle,NIFTY,NIFTY_221_400000_-500000_2024,25.0,2024.0,future_intraday,14460419.0,1807552.0,3585231.0,-499680.0,1333517.0,1.36,-3991620.0,9.03%,2024-09-06
2,Straddle,NIFTY,NIFTY_221_400000_-600000_2024,25.0,2024.0,future_intraday,12659455.0,1582432.0,3585231.0,-499680.0,1486949.0,1.06,-3991620.0,7.91%,2024-09-06
3,Straddle,NIFTY,NIFTY_221_400000_-700000_2024,25.0,2024.0,future_intraday,12659455.0,1582432.0,3585231.0,-499680.0,1486949.0,1.06,-3991620.0,7.91%,2024-09-06


In [12]:
# import subprocess,json
# from datetime import datetime, time, timedelta
# # prefix = '/home/oem/DATA_STORE/OUTPUTS'
# for year in range(2020,2025):
#     for mt in [10,15,20]:
#         year = str(year)
#         prefix = f'/home/cloudcraftz/Cloudcraftz/HFT/alpha_v07/F_Intraday/sig_sample/outputs/NIFTY_VSSMA_CUSTOMER/Quick_Action/{year}/max_trades_{mt}'
#         code_base = 'future_intraday'
#         underlying = 'NIFTY'
#         Strategy = 'Straddle'
#         # year = '2024'
#         weekly_stats = False
#         GET_BLOTTER_SUMMARY = True
#         GET_VSSMA_SUMMARY = True

#         output_summary = process_technical_summary(prefix,code_base, underlying,year,Strategy,lot_size=25)
#         if GET_BLOTTER_SUMMARY:
#             blotter_summary = process_blotter_summary(prefix,weekly_stats)
#             output_summary = pd.merge(output_summary, blotter_summary, on='Short Description')
#         else:
#             pass

#         if GET_VSSMA_SUMMARY:
#             vssma_summary = process_vssma_summary(prefix)
#             output_summary = pd.merge(output_summary, vssma_summary, on='Short Description')
#         else:
#             pass

#         display(output_summary)
#         output_summary.to_csv(os.path.join(prefix,'output.csv'), index=False)

# Bubble Charts: Pure Intraday Strategies

In [ ]:
path = prefix
results = []  
folder_names = os.listdir(path)
folder_names.sort()
for folder_name in folder_names:
    folder_path = os.path.join(path, folder_name)
    # print("process_blotter_summary",folder_name)
    if os.path.isdir(folder_path):
        trade_summary_df = generate_trade_summary_df(folder_path)
        trade_summary_df.rename(inplace=True, columns=rename_dict)
        
        # Calculate the three values and append them to the results list
        if weekly_stats:
            trade_summary_df.index = pd.to_datetime(trade_summary_df.index)
            trade_summary_df = trade_summary_df.resample('W').sum()
            median_trades = trade_summary_df['Daily Strategy Trades'].median()
            max_trades = trade_summary_df['Daily Strategy Trades'].max()
            hedge_trades = trade_summary_df['Daily Hedge Trades'].median()
            results.append([folder_name, median_trades, max_trades, hedge_trades])
        else:
            median_trades = trade_summary_df['Daily Strategy Trades'].median()
            max_trades = trade_summary_df['Daily Strategy Trades'].max()
            hedge_trades = trade_summary_df['Daily Hedge Trades'].median()
            # time_in_market = trade_summary_df['Time in Market (mins)'].mean()/356
            results.append([folder_name, median_trades, max_trades, hedge_trades])

In [ ]:
trade_summary_df.

In [ ]:
trade_summary_df.index = pd.to_datetime(trade_summary_df.index)

In [ ]:
ds = trade_summary_df.copy()

In [ ]:
ds = ds.resample('W').sum()

In [ ]:
ds['Daily Strategy Trades'].median()

In [ ]:
ds['Daily Strategy Trades'].max()

In [ ]:
# !pip install humanize
# !pip install plotly
# !pip install openpyxl
# !pip install -U kaleido

Code to extract technical summary

In [ ]:
selected_columns = [
        'Short Description',
        'Lot Size',
        'Year',
        'Code Base',
        'Total P/L',
        'Avg. Weekly PnL',
        'Max Weekly Gain',
        'Max Weekly Loss',
        'SD Weekly PnL',
        'Weekly Avg. PnL / SD',
        'Peak Drawdown',
        'Ratio (Total P/L / Net Premium)',
        'Daily Peak Margin',
        'Avg. Daily Peak Margin'
    ]

def excel_to_dict(file_path):
    result_dict = {}
    # Read Excel file into a pandas DataFrame
    df = pd.read_excel(file_path)
    # Assuming the first row is a header, we skip it
    for index, row in df.iterrows():
        for i in range(0, len(row), 2):
            if i + 1 < len(row):
                result_dict[row[i]] = row[i + 1]
    return result_dict

def convert_to_number(value):
    try:
        # Remove non-numeric characters and convert to float
        return float(value.replace(',', '').split()[0])
    except ValueError:
        return value  # Return the original value if conversion fails

def process_folder(prefix,ul,y,s):


    main_folder_path = prefix
    print(main_folder_path)

    list_dicts = []

    # Iterate through each folder in the main folder
    folder_names = os.listdir(main_folder_path)
    folder_names.sort()
    for folder_name in folder_names:
        folder_path = os.path.join(main_folder_path, folder_name, 'consolidated_store')
        if os.path.isdir(folder_path):
            # print("Reading files from folder:", folder_name)
            # Iterate through files in the current folder
            for file_name in os.listdir(folder_path):
                if file_name.endswith('.xlsx'):
                    file_path = os.path.join(folder_path, file_name)
                    # Read the YAML file
                    excel_dict = excel_to_dict(file_path)
                    converted_data = {key: convert_to_number(str(value)) for key, value in excel_dict.items()}
                    list_dicts.append(converted_data)
        # break

    

    df = pd.DataFrame(list_dicts)
    df_selected = df[selected_columns]
    display(df)

    # df_selected.to_csv(os.path.join(main_folder_path,'output.csv'), index=False)
    
    # title = f"{ul} {y} {s} Pure Intraday Strategy Time Combinations Comparison"
    # fig = create_bubble_chart(df, title)
    # fig.write_image(os.path.join(main_folder_path,'visualisation.png'))

Plotting Code

In [ ]:
# Create a DataFrame with the provided data
times = {
    "Start Time": ["9:30 AM", "9:30 AM", "9:30 AM", "9:45 AM", "9:45 AM", "9:45 AM", "10:00 AM", "10:00 AM", "10:00 AM"],
    "End Time": ["3:15 PM", "3:30 PM", "3:45 PM"]*3,
}

def create_bubble_chart(data, title):

    df = pd.DataFrame(data)
    for t in times:df[t] = times[t]
    df['Weekly Avg. PnL / SD']*=100

    scaling_factor = math.e  # Adjust as needed

    # Apply scaling factor to the bubble sizes
    df['Scaled P/L'] = df['Total P/L'].rank() ** scaling_factor
    df['Scaled Weekly Avg. PnL / SD'] = df['Weekly Avg. PnL / SD'].rank() ** scaling_factor

    # Create subplots
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Total P/L (USD)", "Weekly Avg. PnL / SD (%)"))

    for i, metric in enumerate([('Total P/L', 'Scaled P/L'), ('Weekly Avg. PnL / SD', 'Scaled Weekly Avg. PnL / SD')]):
        df_metric = df[metric[0]]
        scaled_metric = df[metric[1]]

        fig.add_trace(
            go.Scatter(
                x=df['Start Time'],
                y=df['End Time'],
                mode='markers',
                marker=dict(size=scaled_metric, sizemode='area', sizeref=2.0 * max(scaled_metric) / (50.0 ** 2)),
                showlegend=False
            ),
            row=1, col=i + 1
        )

        # Add annotations
        for j in range(len(df)):
            fig.add_annotation(x=df.iloc[j]['Start Time'], y=df.iloc[j]['End Time'], text=humanize.intword(df_metric.iloc[j]), showarrow=True,row=1, col=i + 1)

        # Update x and y axes titles
        fig.update_xaxes(title_text="Start Time", row=1, col=i + 1)
        fig.update_yaxes(title_text="End Time", row=1, col=i + 1)

    # Update other layout properties
    fig.update_layout(
        width=1600,
        title=title,
        coloraxis_colorbar=dict(title="Colorbar Title")
    )

    # fig.show()
    return fig

# Exmaple Usage:
# title = "NIFTY 2023 Straddle Pure Intraday Strategy Time Combinations Comparison"
# create_bubble_chart(data_nifty_straddle_2023, title)


In [ ]:
import re
def extract_dates(string:str):
    # Define a regular expression pattern to match dates in the format 'YYYY-MM-DD'
    date_pattern = r'\d{4}-\d{2}-\d{2}'
    
    second_dates = []
    log_strings = string.split('\n')
    # print(log_strings)
    
    for log_string in log_strings:
        # Find all matches of dates in the string
        dates = re.findall(date_pattern, log_string)
        
        # Extract the second date if available
        if len(dates) > 1:
            second_date = dates[1]
            second_dates.append(second_date)
    
    return set(second_dates)

In [ ]:
pattern = r'\b(19\d{2}|20\d{2})\b'
match = re.search(pattern, 'SPX_2022_07:00:00_15:30:00')
match

In [ ]:
'SPX_2022_07:00:00_15:30:00'.split('2022')[1][1:]

In [ ]:
pattern = r'\b(20\d{2})\b'

# Function to extract year
def extract_year(text):
    match = re.search(pattern, text)
    if match:
        return match.group(0)
    return None

In [ ]:
re.search(pattern,'SPX_2022_sgdgsd')

In [ ]:
import subprocess,json
from datetime import datetime, time, timedelta
# prefix = '/home/oem/DATA_STORE/OUTPUTS'
prefix = '/home/cloudcraftz/Cloudcraftz/HFT/future_intraday/sig_sample/outputs/SPX/START_END_TIME_OPTIMIZATION/2022'
code_base = 'future_intraday'
underlyings = [
    # 'NIFTY',
    # 'BANKNIFTY',
    'SPX'
]
years = [
    # 2020,
    # 2021,
    # 2022,
    # 2023,
    '']
strategies = [
    # 'STRADDLE',
    # 'STRANGLE',
    # '00050',
    # '01130',
    # 'Baseline'
    ''
]

for ul in underlyings:
    for y in years:
        for s in strategies:
            process_folder(prefix,code_base, str(ul),str(y),str(s))
            # command = f'cat "/home/oem/DATA STORE/OUTPUTS/Pure Intraday Strategies/BANKNIFTY/{y}/STRADDLE/BANKNIFTY_{y}_PURE_INTRADAY_STRADDLE_0920_1515/log/"*.log | grep "CRITICAL"'
            # result = subprocess.run(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, universal_newlines=True)
            # # print(f'error : {result.stdout} len : {len(result.stdout)}')
            # extracted_dates = extract_dates(result.stdout)
            
            # file_path = f'/home/oem/Code/HFT-Options-EIS-Global/datasets/holiday_lists/holidays_{y}.json'
            
            # with open(file_path) as fp:
            #     dates = json.load(fp)
            # # Remove "00:00:00" part from each date
            # dates_without_time = [date.split()[0] for date in dates]
            # holidays = set(dates_without_time)
            
            # print(ul,y)
            # set_diff = extracted_dates - holidays
            # list_diff = sorted(list(set_diff))
            # for d in list_diff:print(d)
            # print('\n')
            # break
    # break

# Log Validator

In [ ]:
# ! cat "/home/oem/DATA STORE/OUTPUTS/Pure Intraday Strategies/BANKNIFTY/2023/STRADDLE/BANKNIFTY_2023_PURE_INTRADAY_STRADDLE_0920_1515/log/"*.log | grep "CRITICAL - intraday data" 
# # > '/home/cloudcraftz/Cloudcraftz/HFT/alpha_v06_dev/sig_sample/outputs/Straddle/SPXW_Straddle_gh_221_2023/log/critical.log'

In [ ]:
# ["2023-01-26 00:00:00", "2023-03-07 00:00:00", "2023-03-30 00:00:00", "2023-04-04 00:00:00", "2023-04-07 00:00:00", "2023-04-14 00:00:00", "2023-05-01 00:00:00", "2023-06-28 00:00:00", "2023-08-15 00:00:00", "2023-09-19 00:00:00", "2023-10-02 00:00:00", "2023-10-24 00:00:00", "2023-11-14 00:00:00", "2023-11-27 00:00:00", "2023-12-25 00:00:00"]